# Huấn luyện PhoBERT — Phân loại chủ đề Tin tức Tiếng Việt
## PhoBERT Fine-tuning Pipeline (train_v4.py)

**Mô hình:** `vinai/phobert-base` (135M tham số)  
**Tác vụ:** Phân loại đa lớp (Multi-class Classification) — 11 chủ đề  
**Kỹ thuật:** R-Drop + Label Smoothing + Class Weights + Sliding Window

---

### Pipeline tổng quan

```
1. Đọc CSV (nhiều encoding)
2. Xây dựng input text (schema tự nhận diện)
3. Làm sạch (clean text)
4. Tách từ tiếng Việt (word segmentation — underthesea)
5. Tăng cường dữ liệu (data augmentation — nlpaug)
6. Tokenization (PhoBERT BPE tokenizer)
7. Load PhoBERT-base
8. Training với CustomTrainer
9. Đánh giá chi tiết
10. Lưu reports + model
```

> **Dataset mẫu:** `mini_test_dataset.csv` (29 bài, 3 class)
> **Dataset đầy đủ:** `data/merged_ultra_clean_data.csv` (225,731 bài, 11 class)

---


## Bước 1: Import thư viện

In [ ]:
import json
import os
import platform
import re
import shutil
import sys
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Kiểm tra GPU
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")


## Bước 2: Cấu hình (Configuration)

In [ ]:
# ======================================================================
# ████  CONFIG — chỉnh sửa tại đây
# ======================================================================

# ── Dataset ──────────────────────────────────────────────────────────
# Dataset mẫu (nhỏ — để test nhanh)
SOURCE_DATA_PATH = Path("mini_test_dataset.csv")

# Dataset đầy đủ (bỏ comment nếu muốn dùng)
# SOURCE_DATA_PATH = Path("data/merged_ultra_clean_data.csv")

# ── Model ───────────────────────────────────────────────────────────
MODEL_NAME = "vinai/phobert-base"

# ── Tiền xử lý ────────────────────────────────────────────────────
# Bật word segmentation (tách từ) bằng underthesea
USE_WORD_SEGMENT = False   # dataset mini đã segment sẵn

# Bật data augmentation bằng nlpaug (tăng dữ liệu cho class ít bài)
USE_AUGMENTATION = False

# ── Tokenizer ───────────────────────────────────────────────────────
MAX_LENGTH = 256    # tokens tối đa
STRIDE     = 128    # stride cho sliding window evaluation

# ── Training ─────────────────────────────────────────────────────────
LEARNING_RATE  = 2e-5
BATCH_SIZE     = 32
NUM_EPOCHS     = 10
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.06
EARLY_STOPPING = 3    # epochs không cải thiện → dừng

# ── Custom loss (R-Drop + Label Smoothing) ─────────────────────────
RDROP_ALPHA      = 2.0     # Cường độ regularization R-Drop
LABEL_SMOOTHING = 0.05     # Label smoothing epsilon

# ── Pilot mode (chạy nhanh) ─────────────────────────────────────────
# Bật True để giảm dữ liệu, chỉ train 1 epoch → test nhanh
PILOT_MODE = True

if PILOT_MODE:
    NUM_EPOCHS   = 1
    BATCH_SIZE   = 16
    MAX_LENGTH   = 192
    STRIDE       = 96

# ── Split ────────────────────────────────────────────────────────────
TEST_SIZE     = 0.2
RANDOM_SEED   = 42

# ── Label mapping ─────────────────────────────────────────────────────
LABEL_NAMES = {
    0:  "giao_duc",
    1:  "kinh_doanh",
    2:  "lao_dong",
    3:  "nhan_ai_cong_dong",
    4:  "phap_luat",
    5:  "suc_khoe_gia_dinh",
    6:  "the_gioi",
    7:  "the_thao",
    8:  "van_hoa_giai_tri",
    9:  "xe_cong_nghe",
    10: "xa_hoi",
}

print("=== CẤU HÌNH ===")
print(f"  Dataset : {SOURCE_DATA_PATH}")
print(f"  Model   : {MODEL_NAME}")
print(f"  Epochs  : {NUM_EPOCHS}")
print(f"  Batch   : {BATCH_SIZE}")
print(f"  Pilot   : {PILOT_MODE}")


## Bước 3: Hàm tiện ích — Đọc CSV & Làm sạch văn bản

In [ ]:
# ======================================================================
# 3A. Đọc CSV với nhiều encoding
# ======================================================================

def read_csv_robust(path: Path) -> pd.DataFrame:
    """Đọc CSV tự động thử nhiều encoding phổ biến."""
    path = Path(path)
    for enc in ["utf-8", "utf-8-sig", "cp1258", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            pass
    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace")


# ======================================================================
# 3B. Làm sạch văn bản tiếng Việt
# ======================================================================

def clean_text(text: str) -> str:
    """
    Làm sạch text:
      1. Chuẩn hóa Unicode (NFC) — chống trùng lặp do NFD/NFC
      2. Xóa HTML tags
      3. Xóa URL và email
      4. Xóa ký tự điều khiển
      5. Chuẩn hóa ký tự lặp lại (!!!! → !)
      6. Chuẩn hóa khoảng trắng
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<[^>]+>", " ", text)                   # HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)          # Email
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)  # Control chars
    text = re.sub(r"([!?.]){3,}", r"\1", text)            # Dấu lặp
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ======================================================================
# 3C. Xây dựng cột input text từ nhiều schema CSV
# ======================================================================

def build_input_text(df: pd.DataFrame) -> pd.Series:
    """
    Tự nhận diện schema CSV và ghép text phù hợp:
      - Schema 1 (mới):  title + description + content
      - Schema 2 (v3):   input_text
      - Schema 3 (v2):   text_segmented / content_segmented
    """
    cols = set(df.columns)

    if "text_segmented" in cols:
        print("  ✓ Dùng cột 'text_segmented' (đã tách từ sẵn)")
        return df["text_segmented"].fillna("").astype(str)

    if "content" in cols:
        parts = []
        if "title" in cols:
            parts.append(df["title"].fillna("").astype(str))
        if "description" in cols:
            parts.append(df["description"].fillna("").astype(str))
        parts.append(df["content"].fillna("").astype(str))
        combined = parts[0]
        for p in parts[1:]:
            combined = combined + ". " + p
        print("  ✓ Dùng schema mới: title + description + content")
        return combined.str.strip()

    if "input_text" in cols:
        series = (df["input_text"]
                  .astype(str)
                  .str.replace(r"\s*\[SEP\]\s*", ". ", regex=True)
                  .str.strip())
        print("  ✓ Dùng cột 'input_text' (đã thay [SEP] → '. ')")
        return series

    if "content_segmented" in cols:
        base = df["content_segmented"].fillna("").astype(str)
        if "title_segmented" in cols:
            print("  ✓ Dùng 'title_segmented + content_segmented'")
            return (df["title_segmented"].fillna("").astype(str) + " " + base).str.strip()
        return base

    for col in ["text_segmented", "text_cleaned", "text_raw", "text"]:
        if col in cols:
            print(f"  ✓ Dùng cột '{col}'")
            return df[col].fillna("").astype(str)

    raise ValueError(
        f"Không tìm thấy cột text phù hợp.\n"
        f"Cần ít nhất: content, input_text, content_segmented, text_segmented.\n"
        f"Cột hiện tại: {list(df.columns)}"
    )


# ======================================================================
# 3D. Tách từ tiếng Việt (Word Segmentation)
# ======================================================================

def segment_text_series(texts: pd.Series, desc: str = "Segmenting") -> pd.Series:
    """Tách từ bằng underthesea."""
    try:
        from underthesea import word_tokenize
    except ImportError:
        print("  ⚠️  underthesea chưa cài — bỏ qua segmentation")
        return texts

    segmented = []
    for text in texts:
        try:
            segmented.append(word_tokenize(text, format="text"))
        except Exception:
            segmented.append(text)
    return pd.Series(segmented, index=texts.index)


print("✓ Các hàm tiện ích đã định nghĩa")


## Bước 4: Đọc & tiền xử lý dữ liệu

In [ ]:
# ======================================================================
# 4A. Đọc dataset
# ======================================================================
print("=" * 60)
print("LOAD DATASET")
print("=" * 60)

if not SOURCE_DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy: {SOURCE_DATA_PATH}")

raw_df = read_csv_robust(SOURCE_DATA_PATH)
print(f"  File  : {SOURCE_DATA_PATH}")
print(f"  Shape : {raw_df.shape}")
print(f"  Cột   : {list(raw_df.columns)}")


# ======================================================================
# 4B. Xây dựng cột text
# ======================================================================
print("\n" + "=" * 60)
print("XÂY DỰNG INPUT TEXT")
print("=" * 60)

if "label" not in raw_df.columns:
    raise ValueError(f"Thiếu cột 'label'. Cột có sẵn: {list(raw_df.columns)}")

raw_df["text"] = build_input_text(raw_df)

# Chuẩn hóa label (hỗ trợ cả số và chuỗi)
_name2id = {v: k for k, v in LABEL_NAMES.items()}

def normalize_label(x):
    if isinstance(x, str):
        x = x.strip().lower()
        if x in _name2id:
            return _name2id[x]
    try:
        val = int(x)
        if val in _name2id.values():
            return val
        return None
    except (ValueError, TypeError):
        return None

raw_df["label"] = raw_df["label"].apply(normalize_label)

# Thống kê phân bố
_label_counts = raw_df["label"].value_counts(dropna=False).sort_index()
print(f"\n  Phân bố label sau chuẩn hóa:")
for lbl, cnt in _label_counts.items():
    name = LABEL_NAMES.get(int(lbl), str(lbl)) if not pd.isna(lbl) else "NaN"
    print(f"    [{int(lbl) if not pd.isna(lbl) else 'NaN':>2}] {name:<22}: {cnt}")


# ======================================================================
# 4C. Làm sạch + giữ lại bài đủ dài
# ======================================================================
print("\n" + "=" * 60)
print("LÀM SẠCH TEXT")
print("=" * 60)

raw_df["text"] = raw_df["text"].apply(clean_text)

source_df = (
    raw_df[["text", "label"]]
    .dropna(subset=["text", "label"])
    .copy()
    .reset_index(drop=True)
)
source_df["label"] = source_df["label"].astype(int)
source_df["text"] = source_df["text"].astype(str).str.strip()

MIN_TEXT_LEN = 20
before = len(source_df)
source_df = source_df[source_df["text"].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)
print(f"  Trước clean : {before} bài")
print(f"  Sau clean    : {len(source_df)} bài (loại {before - len(source_df)} bài quá ngắn)")


## Bước 5: Chia Train / Test + Word Segmentation

In [ ]:
# ======================================================================
# 5A. Train / Test Split (Stratified)
# ======================================================================
print("\n" + "=" * 60)
print("SPLIT TRAIN / TEST")
print("=" * 60)

train_df, test_df = train_test_split(
    source_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=source_df["label"],
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"  Tỷ lệ    : {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} stratified")
print(f"  Train     : {len(train_df)} bài")
print(f"  Test      : {len(test_df)} bài")

print(f"\n  Phân bố label (train):")
for lbl, cnt in sorted(train_df["label"].value_counts().items()):
    name = LABEL_NAMES.get(int(lbl), str(lbl))
    print(f"    [{int(lbl):>2}] {name:<15}: {cnt}")

if PILOT_MODE:
    print(f"\n  ⚡ PILOT MODE: giữ nguyên dữ liệu (mini dataset)")


# ======================================================================
# 5B. Word Segmentation (tùy chọn)
# ======================================================================
print("\n" + "=" * 60)
print("WORD SEGMENTATION")
print("=" * 60)

if USE_WORD_SEGMENT:
    print("  Đang tách từ bằng underthesea...")
    train_df["text"] = segment_text_series(train_df["text"], "Word segmentation")
    print("  ✓ Hoàn tất word segmentation")
else:
    print("  BỎ QUA — dataset đã được segment sẵn")


## Bước 6: Tokenization (PhoBERT BPE)

In [ ]:
# ======================================================================
# 6A. Load PhoBERT tokenizer
# ======================================================================
print("\n" + "=" * 60)
print("TOKENIZATION")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  Tokenizer: {MODEL_NAME}")
print(f"  Vocab size: {tokenizer.vocab_size:,}")
print(f"  Max length: {MAX_LENGTH} tokens")


# ======================================================================
# 6B. Hàm tokenize cho train (lấy chunk đầu tiên)
# ======================================================================

def tokenize_train(examples):
    """Tokenize cho training — lấy 256 tokens đầu tiên."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


# ======================================================================
# 6C. Hàm Sliding Window cho evaluation
# ======================================================================

def sliding_window_tokenize(examples, indices):
    """
    Chia văn bản dài thành nhiều chunks chồng lấp:
      - chunk_size = MAX_LENGTH - 2 (chừa CLS và SEP)
      - stride = STRIDE (128 tokens → 50% overlap)

    Mỗi chunk → một sample riêng trong eval.
    sample_idx = chỉ số bài gốc (để aggregate logits).
    """
    all_input_ids, all_attention_mask = [], []
    all_labels, all_sample_idx = [], []
    chunk_size = MAX_LENGTH - 2

    for i, (text, label) in enumerate(zip(examples["text"], examples["label"])):
        idx = indices[i]
        enc = tokenizer(text, truncation=False, add_special_tokens=False)
        token_ids = enc["input_ids"]
        start = 0
        while start < len(token_ids):
            chunk = token_ids[start:start + chunk_size]
            chunk_with_special = (
                [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
            )
            all_input_ids.append(chunk_with_special)
            all_attention_mask.append([1] * len(chunk_with_special))
            all_labels.append(label)
            all_sample_idx.append(idx)
            if start + chunk_size >= len(token_ids):
                break
            start += STRIDE

    return {
        "input_ids":      all_input_ids,
        "attention_mask": all_attention_mask,
        "label":          all_labels,
        "sample_idx":     all_sample_idx,
    }


# ======================================================================
# 6D. Áp dụng tokenization
# ======================================================================

train_hf = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
test_hf  = Dataset.from_pandas(test_df[["text", "label"]],  preserve_index=False)

train_hf = train_hf.map(
    tokenize_train, batched=True,
    remove_columns=["text"],
    desc="Tokenize train",
)
test_hf = test_hf.map(
    sliding_window_tokenize, batched=True,
    with_indices=True,
    remove_columns=["text"],
    desc="Sliding window eval",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"  Train: {len(train_df)} bài → {len(train_hf)} samples")
print(f"  Test : {len(test_df)} bài → {len(test_hf)} chunks")
print(f"         (TB {len(test_hf)/len(test_df):.2f} chunks/bài)")


## Bước 7: Tính Class Weights (xử lý dữ liệu mất cân bằng)

In [ ]:
# ======================================================================
# Xác định số labels và label mapping
# ======================================================================
labels     = sorted(train_df["label"].unique().tolist())
num_labels = len(labels)

id2label = {i: LABEL_NAMES.get(i, str(i)) for i in labels}
label2id = {name: idx for idx, name in id2label.items()}

print(f"  Số labels : {num_labels}")
print(f"  Mapping   : {id2label}")


# ======================================================================
# Tính Class Weights — xử lý imbalanced dataset
# ======================================================================
# Công thức: w_k = N / (K × n_k)
# Class ít bài → weight cao → loss contribution lớn → model chú ý hơn

_label_arr = train_df["label"].values
_cw = compute_class_weight("balanced", classes=np.array(labels), y=_label_arr)

_max_w = _cw.max()
_min_w = _cw.min()

if _max_w / _min_w > 1.5:
    class_weights = torch.tensor(_cw, dtype=torch.float32)
    print(f"\n  Class Weights (imbalanced):")
    for i, lbl in enumerate(labels):
        print(f"    [{lbl}] {id2label[lbl]:<15}: {_cw[i]:.3f}")
else:
    class_weights = None
    print(f"\n  Dataset cân bằng → class_weights = None")


## Bước 8: Load PhoBERT + Custom Trainer

In [ ]:
# ======================================================================
# 8A. Load PhoBERT-base và gắn head phân loại mới
# ======================================================================
print("\n" + "=" * 60)
print("LOAD MODEL")
print("=" * 60)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print(f"  Model  : {MODEL_NAME}")
print(f"  Labels : {num_labels}")
print(f"  Chờ load weight...")


# ======================================================================
# 8B. Custom Trainer — R-Drop + Label Smoothing + Class Weights
# ======================================================================

class CustomTrainer(Trainer):
    """
    Trainer tùy chỉnh tích hợp 3 kỹ thuật regularization:

    1. Cross-Entropy với Label Smoothing (ε = 0.05)
       → Giảm overconfidence, tăng tổng quát hóa

    2. R-Drop (α = 2.0)
       → 2 forward pass với 2 dropout mask khác nhau
       → Minimize Symmetric KL Divergence giữa 2 phân phối
       → Ép model ổn định bất kể dropout

    3. Class Weights (tùy chọn)
       → Cân bằng loss cho class thiểu số
    """

    def __init__(self, *args,
                 class_weights=None,
                 label_smoothing=0.1,
                 rdrop_alpha=4.0,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights   = class_weights
        self.label_smoothing = label_smoothing
        self.rdrop_alpha     = rdrop_alpha

    def _ce_loss(self, logits, labels):
        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device),
                label_smoothing=self.label_smoothing,
            )
        else:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=self.label_smoothing)
        return loss_fct(logits, labels)

    @staticmethod
    def _sym_kl(p_logits, q_logits):
        """Symmetric KL: 0.5 × (KL(P||Q) + KL(Q||P))"""
        p = F.softmax(p_logits, dim=-1)
        q = F.softmax(q_logits, dim=-1)
        kl_pq = F.kl_div(F.log_softmax(p_logits, dim=-1), q, reduction="batchmean")
        kl_qp = F.kl_div(F.log_softmax(q_logits, dim=-1), p, reduction="batchmean")
        return 0.5 * (kl_pq + kl_qp)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        inputs.pop("sample_idx", None)

        if self.rdrop_alpha > 0.0:
            # R-Drop: 2 forward pass → 2 phân phối logits
            out1 = model(**inputs); logits1 = out1.logits
            out2 = model(**inputs); logits2 = out2.logits
            ce_loss = 0.5 * (self._ce_loss(logits1, labels) + self._ce_loss(logits2, labels))
            kl_loss = self._sym_kl(logits1, logits2)
            loss = ce_loss + self.rdrop_alpha * kl_loss
            return (loss, out1) if return_outputs else loss
        else:
            out = model(**inputs)
            loss = self._ce_loss(out.logits, labels)
            return (loss, out) if return_outputs else loss


print("  ✓ CustomTrainer định nghĩa xong")


## Bước 9: Hàm đánh giá (Sliding Window Aggregation)

In [ ]:
# ======================================================================
# Metrics — Aggregate logits từ nhiều chunks về bài gốc
# ======================================================================

_test_sample_idx_global = np.array(test_hf["sample_idx"])


def compute_metrics(eval_pred):
    """
    Tính Accuracy và F1-Score sau khi aggregate logits:
      - Bài dài → nhiều chunks → nhiều logits
      - Logits trung bình → phân phối xác suất cuối cùng
      - argmax → dự đoán class
    """
    logits, labels = eval_pred

    n_samples = int(_test_sample_idx_global.max()) + 1
    n_classes = logits.shape[1]
    agg_logits  = np.zeros((n_samples, n_classes), dtype=np.float64)
    agg_counts  = np.zeros(n_samples, dtype=np.int32)
    true_labels = np.zeros(n_samples, dtype=np.int64)

    for ci, si in enumerate(_test_sample_idx_global):
        agg_logits[si] += logits[ci]
        agg_counts[si] += 1
        true_labels[si] = labels[ci]

    agg_logits /= np.maximum(agg_counts[:, None], 1)
    preds = np.argmax(agg_logits, axis=1)

    return {
        "accuracy": float(accuracy_score(true_labels, preds)),
        "f1":       float(f1_score(true_labels, preds, average="weighted")),
    }


print("  ✓ compute_metrics định nghĩa xong")


## Bước 10: Cấu hình TrainingArguments

In [ ]:
# ======================================================================
# TrainingArguments — toàn bộ hyperparameter
# ======================================================================

# Tạo thư mục output
from datetime import datetime
_run_id   = datetime.now().strftime("%Y%m%d-%H%M%S")
OUT_DIR   = Path(f"phobert-topic-model-v4/runs/run-{_run_id}")
REPORT_DIR = Path(f"phobert-topic-model-v4/reports/run-{_run_id}")
LATEST_DIR = Path("phobert-topic-model-v4/latest")
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = str(OUT_DIR),
    eval_strategy               = "epoch",
    logging_strategy            = "epoch",
    save_strategy               = "epoch",
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = 1,
    num_train_epochs            = NUM_EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    report_to                   = "none",
    fp16                        = torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    dataloader_pin_memory       = torch.cuda.is_available(),
    save_total_limit            = 2,
    dataloader_num_workers      = 0,   # Windows: worker=0 tránh lỗi spawn
)

print("=== TRAINING ARGUMENTS ===")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  R-Drop α       : {RDROP_ALPHA}")
print(f"  Label smooth ε : {LABEL_SMOOTHING}")
print(f"  Warmup         : {WARMUP_RATIO*100:.0f}%")
print(f"  Early stopping : {EARLY_STOPPING} epochs")
print(f"  Output dir     : {OUT_DIR}")


## Bước 11: Callbacks (Progress + Sanity Check + Early Stopping)

In [ ]:
# ======================================================================
# Progress Callback — hiển thị tqdm bar
# ======================================================================

class ProgressCallback(TrainerCallback):
    def __init__(self):
        self._bar = None
        self._last_step = 0

    def on_train_begin(self, args, state, control, **kwargs):
        total = int(state.max_steps or 0)
        if total > 0:
            self._bar = tqdm(total=total, desc="Training", unit="step")
        self._last_step = 0

    def on_step_end(self, args, state, control, **kwargs):
        if self._bar is None:
            return
        delta = max(0, int(state.global_step) - self._last_step)
        if delta:
            self._bar.update(delta)
            self._last_step = int(state.global_step)
        if state.epoch is not None:
            self._bar.set_postfix(epoch=f"{state.epoch:.1f}")

    def on_train_end(self, args, state, control, **kwargs):
        if self._bar:
            self._bar.close()


# ======================================================================
# Sanity Check Callback — phát hiện model collapse
# ======================================================================

class PredictionSanityCallback(TrainerCallback):
    """
    Sau mỗi epoch, kiểm tra phân phối dự đoán.
    Nếu >50% dự đoán rơi vào 1 class → model có thể collapse.
    """
    def __init__(self, eval_dataset, id2label):
        self._eval_dataset = eval_dataset
        self._id2label    = id2label

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        device = next(model.parameters()).device
        from torch.utils.data import DataLoader
        collator = DataCollatorWithPadding(tokenizer=tokenizer)

        subset = self._eval_dataset.select(range(min(512, len(self._eval_dataset))))
        loader = DataLoader(subset, batch_size=32, collate_fn=collator)

        all_preds = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()
                         if k in ("input_ids", "attention_mask")}
                logits = model(**batch).logits
                all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        model.train()

        unique, counts = np.unique(all_preds, return_counts=True)
        total = len(all_preds)
        dist  = {self._id2label.get(int(u), str(u)): int(c) for u, c in zip(unique, counts)}
        dominant = counts.max() / total

        print(f"\n  ── Sanity (epoch {state.epoch:.0f}) ──")
        print(f"     Active classes: {len(unique)}/{len(self._id2label)}")
        if dominant > 0.5:
            print(f"  ⚠️  CẢNH BÁO: {dominant*100:.0f}% rơi vào 1 class — model có thể collapse!")
        for name, cnt in sorted(dist.items(), key=lambda x: -x[1])[:6]:
            bar = "█" * int(cnt / total * 25)
            print(f"     {name:<20}: {cnt:>4} ({cnt/total*100:5.1f}%) {bar}")


print("  ✓ Callbacks định nghĩa xong")


## Bước 12: Huấn luyện! (trainer.train())

In [ ]:
# ======================================================================
# Khởi tạo CustomTrainer và huấn luyện
# ======================================================================
print("\n" + "=" * 60)
print("HUẤN LUYỆN")
print("=" * 60)
print(f"  Model     : {MODEL_NAME}")
print(f"  Train     : {len(train_df)} bài")
print(f"  Test      : {len(test_df)} bài")
print(f"  Epochs    : {NUM_EPOCHS}")
print(f"  Batch     : {BATCH_SIZE}")
print(f"  R-Drop α  : {RDROP_ALPHA}")
print(f"  GPU       : {'CUDA ✓' if torch.cuda.is_available() else 'CPU'}")
print("=" * 60)

trainer = CustomTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_hf,
    eval_dataset     = test_hf,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = tokenizer,
    class_weights    = class_weights,
    label_smoothing  = LABEL_SMOOTHING,
    rdrop_alpha      = RDROP_ALPHA,
)

trainer.add_callback(ProgressCallback())
trainer.add_callback(
    EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING,
        early_stopping_threshold=0.001,
    )
)
trainer.add_callback(PredictionSanityCallback(test_hf, id2label))

# BẮT ĐẦU HUẤN LUYỆN
trainer.train()

# Đánh giá cuối
eval_metrics = trainer.evaluate()
print(f"\n  ✓ Hoàn tất! Eval metrics:")
for k, v in eval_metrics.items():
    print(f"    {k}: {v}")


## Bước 13: Đánh giá chi tiết + Phân tích lỗi

In [ ]:
# ======================================================================
# Đánh giá chi tiết — Sliding Window Aggregation
# ======================================================================
print("\n" + "=" * 60)
print("ĐÁNH GIÁ CHI TIẾT")
print("=" * 60)

pred_output = trainer.predict(test_hf)
chunk_logits = pred_output.predictions
chunk_sample_idx = np.array(test_hf["sample_idx"])

n_samples  = len(test_df)
agg_logits = np.zeros((n_samples, num_labels), dtype=np.float64)
agg_counts = np.zeros(n_samples, dtype=np.int32)
for ci, si in enumerate(chunk_sample_idx):
    agg_logits[si] += chunk_logits[ci]
    agg_counts[si] += 1
agg_logits /= agg_counts[:, None]

# Softmax → xác suất
probs       = torch.softmax(torch.tensor(agg_logits, dtype=torch.float32), dim=-1)
predictions = probs.argmax(dim=-1).numpy()
confidences = probs.max(dim=-1).values.numpy()
true_labels = test_df["label"].values

target_names = [id2label.get(i, str(i)) for i in range(num_labels)]
print(classification_report(
    true_labels, predictions,
    labels=list(range(num_labels)),
    target_names=target_names,
    zero_division=0,
))

accuracy = accuracy_score(true_labels, predictions)
print(f"  ★ Accuracy: {accuracy*100:.2f}%")
print(f"  ★ F1 (weighted): {f1_score(true_labels, predictions, average='weighted'):.4f}")


# ======================================================================
# Phân tích lỗi
# ======================================================================
analysis_df = pd.DataFrame({
    "text":          test_df["text"].values,
    "true_label":    true_labels,
    "true_name":     [id2label.get(int(l), str(l)) for l in true_labels],
    "pred_label":    predictions,
    "pred_name":     [id2label.get(int(p), str(p)) for p in predictions],
    "confidence":    confidences,
    "correct":       true_labels == predictions,
})
wrong_df = analysis_df[~analysis_df["correct"]].copy()

print(f"\n  Đúng : {analysis_df['correct'].sum()}/{len(analysis_df)}")
print(f"  Sai  : {len(wrong_df)}")

# Lỗi confidence cao → có thể label sai
HIGH_CONF = 0.8
hc_errors = wrong_df[wrong_df["confidence"] >= HIGH_CONF]
if len(hc_errors):
    print(f"\n  ⚠️  Lỗi confidence cao (≥{HIGH_CONF*100:.0f}%): {len(hc_errors)}")
    print("     → Có thể data bị gán nhãn sai!")

# Top confusion pairs
cm = confusion_matrix(true_labels, predictions, labels=list(range(num_labels)))
labels_sorted = sorted(analysis_df["true_label"].unique())
pairs = sorted(
    [{"true": r, "pred": c, "n": int(cm[i, j])}
     for i, r in enumerate(labels_sorted)
     for j, c in enumerate(labels_sorted) if i != j and cm[i, j] > 0],
    key=lambda x: x["n"], reverse=True,
)
print(f"\n  Top 5 cặp nhầm lẫn:")
for rank, p in enumerate(pairs[:5], 1):
    print(f"    {rank}. {id2label[p['true']]} → {id2label[p['pred']]}: {p['n']} lần")


## Bước 14: Vẽ Confusion Matrix Heatmap

In [ ]:
# ======================================================================
# Confusion Matrix Heatmap
# ======================================================================
import seaborn as sns

label_names_sorted = [id2label.get(l, str(l)) for l in labels_sorted]

plt.figure(figsize=(max(8, num_labels), max(6, num_labels - 2)))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names_sorted,
    yticklabels=label_names_sorted,
)
plt.title(f"Confusion Matrix — Accuracy {accuracy*100:.2f}%", fontsize=13, pad=12)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(REPORT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"  ✓ Lưu: {REPORT_DIR / 'confusion_matrix.png'}")


## Bước 15: Lưu Reports (CSV + JSON)

In [ ]:
# ======================================================================
# Lưu các file báo cáo
# ======================================================================

# Confusion matrix CSV
pd.DataFrame(cm, index=label_names_sorted, columns=label_names_sorted).to_csv(
    REPORT_DIR / "confusion_matrix.csv", encoding="utf-8-sig"
)

# Tất cả lỗi
wrong_df[["true_name", "pred_name", "confidence", "text"]].sort_values(
    "confidence", ascending=False
).to_csv(REPORT_DIR / "errors_all.csv", index=False, encoding="utf-8-sig")

# Lỗi confidence cao
if len(hc_errors):
    hc_errors[["true_name", "pred_name", "confidence", "text"]].sort_values(
        "confidence", ascending=False
    ).to_csv(REPORT_DIR / "errors_high_confidence.csv", index=False, encoding="utf-8-sig")

# Classification report JSON
report_dict = classification_report(
    true_labels, predictions,
    labels=list(range(num_labels)),
    target_names=target_names,
    zero_division=0,
    output_dict=True,
)
with open(REPORT_DIR / "classification_report.json", "w", encoding="utf-8") as f:
    json.dump(report_dict, f, indent=2, ensure_ascii=False)

# Lỗi theo category
err_by_cat = (
    wrong_df.groupby("true_name")
    .agg(error_count=("true_name", "count"), avg_conf=("confidence", "mean"))
)
err_by_cat["total"] = analysis_df.groupby("true_name").size()
err_by_cat["error_rate_%"] = (err_by_cat["error_count"] / err_by_cat["total"] * 100).round(2)
err_by_cat.sort_values("error_count", ascending=False).to_csv(
    REPORT_DIR / "errors_by_category.csv", encoding="utf-8-sig"
)

print("=== FILES SAVED ===")
print(f"  {REPORT_DIR / 'confusion_matrix.csv'}")
print(f"  {REPORT_DIR / 'confusion_matrix.png'}")
print(f"  {REPORT_DIR / 'errors_all.csv'}")
print(f"  {REPORT_DIR / 'classification_report.json'}")
print(f"  {REPORT_DIR / 'errors_by_category.csv'}")


## Bước 16: Lưu Model + Inference Benchmark

In [ ]:
# ======================================================================
# Lưu model và tokenizer
# ======================================================================
print("\n" + "=" * 60)
print("LƯU MODEL")
print("=" * 60)

# Lưu checkpoint
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Ghi đè "latest"
if LATEST_DIR.exists():
    shutil.rmtree(LATEST_DIR)
shutil.copytree(OUT_DIR, LATEST_DIR)

# Label map
with open(OUT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, indent=2, ensure_ascii=False)

# Training summary
summary = {
    "version":         "v4",
    "model_name":      MODEL_NAME,
    "source_data":     str(SOURCE_DATA_PATH),
    "num_labels":      num_labels,
    "id2label":        id2label,
    "train_samples":   len(train_df),
    "test_samples":    len(test_df),
    "accuracy":        float(accuracy),
    "eval_f1":         float(eval_metrics.get("eval_f1", 0)),
    "rdrop_alpha":     RDROP_ALPHA,
    "label_smoothing": LABEL_SMOOTHING,
    "class_weights":    class_weights is not None,
}
with open(REPORT_DIR / "training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"  ✓ Model lưu: {OUT_DIR}")
print(f"  ✓ Latest   : {LATEST_DIR}")


# ======================================================================
# Inference Benchmark — đo tốc độ suy luận
# ======================================================================
print("\n" + "=" * 60)
print("INFERENCE BENCHMARK")
print("=" * 60)

device = next(model.parameters()).device
sample_text = (
    "Đội tuyển Việt Nam thi đấu xuất sắc tại giải vô địch Đông Nam Á và giành chiến thắng."
)

model.eval()
# Warmup
for _ in range(10):
    enc = tokenizer(sample_text, return_tensors="pt", max_length=256, truncation=True, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        _ = model(**enc)

# Measure
times = []
for _ in range(100):
    enc = tokenizer(sample_text, return_tensors="pt", max_length=256, truncation=True, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(**enc)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

print(f"  Device    : {device}")
print(f"  Mean      : {np.mean(times):.2f} ms")
print(f"  P50       : {np.percentile(times, 50):.2f} ms")
print(f"  P95       : {np.percentile(times, 95):.2f} ms")

print("\n" + "=" * 60)
print(f"✅ HOÀN TẤT — Accuracy: {accuracy*100:.2f}% — F1: {eval_metrics.get('eval_f1', 0):.4f}")
print("=" * 60)
